# Training and validation dataset

## Generating the data

In [1]:
import numpy as np
import math
from matplotlib import pyplot as plt
import random
import pickle
import os

In [2]:
def my_func(x, y, a, b):
    z = (-1 * b) ** -1 * x * ((np.log((y / x) - 1)) - a)
    z_positive = max(z, 0)
    return round(z_positive, 3)

def min_max_scale(min_X, max_X, X):
    return [(x - min_X) / (max_X - min_X) for x in X]

def inv_min_max_scale(min_X, max_X, scaled_X):
    return [x * (max_X - min_X) + min_X for x in scaled_X]

def generate_training_data(num_points):

    input_data = []
    output_data = []
    random.seed(12)
    for point in range(num_points):
        a = round(random.uniform(4, 8), 3)
        b = round(random.uniform(0.25, 2) , 3)
        y = round(random.uniform(0.1, 20) , 3)
        x = round(random.uniform(0.1 * 0.98, 0.98 * y) , 3)
        input_data.append([x, y, a, b])
        output_data.append([my_func(x, y, a, b)])

    return np.array(input_data), np.array(output_data)

In [3]:
num_points = 300000
input_data, output_data = generate_training_data(num_points)

In [4]:
var_name = ["x", "y", "a", "b"]
min_max_scale_dict = {var_name[i]: {"min": min(input_data.transpose()[i]), "max": max(input_data.transpose()[i])} for i in range(4)}

# Proper minmax scaler
min_max_scale_dict["z"] = {"min": min(output_data)[0], "max": max(output_data)[0]}

# # Replacing max value with average value
# min_max_scale_dict["z"] = {"min": min(output_data)[0], "max": round(np.mean(output_data), 3)}

# with open(os.path.join("optimization input", "nn_scaler.pickle"), "wb") as f:
#     pickle.dump(min_max_scale_dict, f)

In [5]:
# Scale the data

scaled_input_data = np.array([min_max_scale(min_max_scale_dict[var_name[i]]["min"], 
                                            min_max_scale_dict[var_name[i]]["max"], input_data.transpose()[i])
                              for i in range(4)
                             ]
                            ).transpose()
scaled_output_data = np.array(min_max_scale(min_max_scale_dict["z"]["min"], 
                                            min_max_scale_dict["z"]["max"], output_data.transpose())).transpose()

In [6]:
min_max_scale_dict

{'x': {'min': 0.098, 'max': 19.541},
 'y': {'min': 0.1, 'max': 20.0},
 'a': {'min': 4.0, 'max': 8.0},
 'b': {'min': 0.25, 'max': 2.0},
 'z': {'min': 0.0, 'max': 741.868}}

## Split the data into training and validation

In [7]:
from sklearn.model_selection import train_test_split

In [8]:
def train_val_splitter(input_data, output_data, val_ratio):

    input_train, input_val, output_train, output_val = train_test_split(scaled_input_data, scaled_output_data, test_size=val_ratio, random_state=42)
    
    return (input_train, output_train), (input_val, output_val)

In [9]:
train_data, val_data = train_val_splitter(input_data, output_data, 0.20)

In [10]:
train_data[0].shape

(240000, 4)

# Train the neural networks

In [19]:
from tensorflow.keras.models import Sequential, Model
from keras import Input
from tensorflow.keras.layers import Dense, Add, Multiply, Activation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.constraints import non_neg
from tensorflow import random

## Setup the neural network parameters

In [12]:
layer_neurons_dict = {"1_layer": [40], "2_layer": [20, 20], "4_layer": [5, 15, 15, 5], 
                      "5_layer": [2, 8, 20, 8, 2], "6_layer": [2, 8, 10, 10, 8, 2]}
layer_activation = "relu"
input_dim = np.shape(train_data[0])[1]
output_dim = np.shape(train_data[1])[1]
output_activation = "linear"
epochs = 1000
batch_size = 1000

In [13]:
import os

def makedir(path):
    if not os.path.exists(path):
        os.mkdir(path)
    return None

In [14]:
current_path = os.getcwd()
cwd = os.path.dirname(current_path)
unn_savepath = os.path.join(cwd, "NN length models", "Unconstrained model")
cnn_savepath = os.path.join(cwd, "NN length models", "Constrained model")
ficnn_savepath = os.path.join(cwd, "NN length models", "FICNN model")
paths = [unn_savepath, cnn_savepath, ficnn_savepath]

for path in paths:
    makedir(path)

## Unconstrained neural network training

In [18]:
%%time

unn_history_vals = {}
for layers, layer_neurons in list(layer_neurons_dict.items())[:-1]:
    random.set_seed(42)
    nn_model = Sequential([Input(shape=(input_dim, ))] 
                          + [Dense(layer_neurons[i], activation=layer_activation) for i in range(len(layer_neurons))] 
                          + [Dense(output_dim, activation=output_activation)]
                         )
    optimizer = Adam(learning_rate=0.0001)
    nn_model.compile(loss="mse", metrics=["mae"], optimizer=optimizer)
    print(nn_model.summary())
    history = nn_model.fit(train_data[0], train_data[1], validation_data=val_data, epochs=epochs, batch_size=batch_size, verbose=0)
    unn_history_vals[layers] = history
    # nn_model.save(os.path.join(unn_savepath, layers + "_Base_NN.keras"))

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_14 (Dense)                     │ (None, 40)                  │             200 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_15 (Dense)                     │ (None, 1)                   │              41 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 241 (964.00 B)

 Trainable params: 241 (964.00 B)

 Non-trainable params: 0 (0.00 B)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_24 (Dense)                     │ (None, 2)                   │              10 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_25 (Dense)                     │ (None, 8)                   │              24 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_26 (Dense)                     │ (None, 20)                  │             180 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_27 (Dense)                     │ (None, 8)                   │             168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_28 (Dense)                     │ (None, 2)                   │              18 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_29 (Dense)                     │ (None, 1)                   │               3 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 403 (1.57 KB)

 Trainable params: 403 (1.57 KB)

 Non-trainable params: 0 (0.00 B)

None
CPU times: total: 38min 57s
Wall time: 24min 10s


In [19]:
# with open(os.path.join(unn_savepath, "history_dict_6.pickle"), "wb") as f:
#     pickle.dump(unn_history_vals, f)

## Constrained neural network training

In [20]:
%%time

cnn_history_vals = {}
for layers, layer_neurons in list(layer_neurons_dict.items())[:-1]:
    random.set_seed(42)
    cnn_model = Sequential([Input(shape=(input_dim, )), Dense(layer_neurons[0], activation=layer_activation)] 
                          + [Dense(layer_neurons[i], activation=layer_activation, kernel_constraint=non_neg) for i in range(1, len(layer_neurons))] 
                          + [Dense(output_dim, activation=output_activation, kernel_constraint=non_neg)]
                         )
    optimizer_cnn = Adam(learning_rate=0.0001)
    cnn_model.compile(loss="mse", metrics=["mae"], optimizer=optimizer_cnn)
    print(cnn_model.summary())
    cnn_history = cnn_model.fit(train_data[0], train_data[1], validation_data=val_data, epochs=epochs, batch_size=batch_size, verbose=0)
    cnn_history_vals[layers] = cnn_history
    # cnn_model.save(os.path.join(cnn_savepath, layers + "_Base_NN.keras"))

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_30 (Dense)                     │ (None, 40)                  │             200 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_31 (Dense)                     │ (None, 1)                   │              41 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 241 (964.00 B)

 Trainable params: 241 (964.00 B)

 Non-trainable params: 0 (0.00 B)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_40 (Dense)                     │ (None, 2)                   │              10 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_41 (Dense)                     │ (None, 8)                   │              24 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_42 (Dense)                     │ (None, 20)                  │             180 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_43 (Dense)                     │ (None, 8)                   │             168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_44 (Dense)                     │ (None, 2)                   │              18 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_45 (Dense)                     │ (None, 1)                   │               3 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 403 (1.57 KB)

 Trainable params: 403 (1.57 KB)

 Non-trainable params: 0 (0.00 B)

None
CPU times: total: 33min 31s
Wall time: 20min 30s


In [17]:
# with open(os.path.join(cnn_savepath, "history_dict_6.pickle"), "wb") as f:
#     pickle.dump(cnn_history_vals, f)

## FICNN model

In [27]:
def get_layer_data(model):
    weights_dict = {}
    for index, layer in enumerate(model.layers):
        weights = layer.get_weights()
        if weights:  # skip layers without weights
            layer_num = layer.name.split("_")[-1]
            prefix = layer.name.rsplit("_", 1)[0]
            if layer_num not in weights_dict.keys():
                weights_dict[layer_num] = {}
            weights_dict[layer_num][prefix] = weights 
    return weights_dict

In [31]:
%%time
ficnn_history_vals = {}
for layers, layer_neurons in list(layer_neurons_dict.items())[-1:]:
    random.set_seed(42)
    nn_length = len(layer_neurons)
    x_set = Input(shape=(input_dim, ), name="input_set")
    
    h = Dense(layer_neurons[0], activation=layer_activation, name="W_0")(x_set)
    
    for i in range(1, len(layer_neurons)):
        h_temp = Dense(layer_neurons[i], kernel_constraint=non_neg, name="W_pos_"+str(i))(h)
        x_temp = Dense(layer_neurons[i], name="W_"+str(i))(x_set)
        h_new = Add(name="layer_sum_"+str(i))([x_temp, h_temp])
        h = Activation(layer_activation, name="layer_output_"+str(i))(h_new)
    
    h_temp = Dense(output_dim, kernel_constraint=non_neg, name="W_pos_"+str(nn_length))(h)
    x_temp = Dense(output_dim, name="W_"+str(nn_length))(x_set)
    
    y = Add(name="output_layer_temp")([x_temp, h_temp])
    y = Activation(output_activation, name="output")(y)
    
    ficnn_model = Model(x_set, y)
    optimizer_ficnn = Adam(learning_rate=0.0001)
    ficnn_model.compile(loss="mse", metrics=["mae"], optimizer=optimizer_ficnn)
    print(ficnn_model.summary())
    ficnn_history = ficnn_model.fit(train_data[0], train_data[1], validation_data=val_data, epochs=epochs, batch_size=batch_size, verbose=0)
    ficnn_history_vals[layers] = ficnn_history
    ficnn_model.save(os.path.join(ficnn_savepath, layers + "_Base_NN.keras"))
    weights_dict = get_layer_data(ficnn_model)
    with open(os.path.join(ficnn_savepath, layers + "_Base_NN.pickle"), "wb") as f:
        pickle.dump(weights_dict, f)

Model: "functional_9"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_set           │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ W_0 (Dense)         │ (None, 2)         │         10 │ input_set[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ W_1 (Dense)         │ (None, 8)         │         40 │ input_set[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ W_pos_1 (Dense)     │ (None, 8)         │         24 │ W_0[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_sum_1 (Add)   │ (None, 8)         │          0 │ W_1[0][0],        │
│                     │                   │            │ W_pos_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_output_1      │ (None, 8)         │          0 │ layer_sum_1[0][0] │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ W_2 (Dense)         │ (None, 10)        │         50 │ input_set[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ W_pos_2 (Dense)     │ (None, 10)        │         90 │ layer_output_1[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_sum_2 (Add)   │ (None, 10)        │          0 │ W_2[0][0],        │
│                     │                   │            │ W_pos_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_output_2      │ (None, 10)        │          0 │ layer_sum_2[0][0] │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ W_3 (Dense)         │ (None, 10)        │         50 │ input_set[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ W_pos_3 (Dense)     │ (None, 10)        │        110 │ layer_output_2[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_sum_3 (Add)   │ (None, 10)        │          0 │ W_3[0][0],        │
│                     │                   │            │ W_pos_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_output_3      │ (None, 10)        │          0 │ layer_sum_3[0][0] │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ W_4 (Dense)         │ (None, 8)         │         40 │ input_set[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ W_pos_4 (Dense)     │ (None, 8)         │         88 │ layer_output_3[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_sum_4 (Add)   │ (None, 8)         │          0 │ W_4[0][0],        │
│                     │                   │            │ W_pos_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_output_4      │ (None, 8)         │          0 │ layer_sum_4[0][0] │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ W_5 (Dense)         │ (None, 2)         │         10 │ input_set[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ W_pos_5 (Dense)     │ (None, 2)         │         18 │ layer_output_4[0… │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 538 (2.10 KB)

 Trainable params: 538 (2.10 KB)

 Non-trainable params: 0 (0.00 B)

None
CPU times: total: 1min 45s
Wall time: 7min 5s


In [32]:
with open(os.path.join(ficnn_savepath, "history_dict_6.pickle"), "wb") as f:
    pickle.dump(ficnn_history_vals, f)